# Data cleaning and preprocessing

### **Load raw dataset**

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/online_retail_II.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### **Create a working copy**

In [2]:
clean_df = df.copy()

In [3]:
missing_values = clean_df.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


checking for customer id if its whole numbers

In [4]:
customer_ids = clean_df["Customer ID"].dropna()

non_integer_ids = customer_ids[
    customer_ids % 1 != 0
]

print("Non-integer Customer IDs:", len(non_integer_ids))
print(non_integer_ids.head(20))

Non-integer Customer IDs: 0
Series([], Name: Customer ID, dtype: float64)


### **Convert data types**

In [5]:
clean_df["InvoiceDate"] = pd.to_datetime(
    clean_df["InvoiceDate"],
    errors="coerce"
)

clean_df["Invoice"] = clean_df["Invoice"].astype(str)

clean_df["StockCode"] = clean_df["StockCode"].astype(str)

clean_df["Customer ID"] = clean_df["Customer ID"].astype("Int64")

### **Remove exact duplicates**

In [6]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates()

after = len(clean_df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 1067371
Rows after: 1033036
Duplicates removed: 34335


### **Identify cancelled transactions**

In [7]:
clean_df["IsCancelled"] = (
    clean_df["Invoice"].str.startswith("C")
)

### **Remove negative price adjustments records**

In [8]:
before = len(clean_df)

clean_df = clean_df[
    clean_df["Price"] >= 0
].copy()

print("Negative price records removed:", before - len(clean_df))

Negative price records removed: 5


### **Create valid sales dataset**

In [9]:
sales_df = clean_df[
    (~clean_df["IsCancelled"]) &
    (clean_df["Quantity"] > 0) &
    (clean_df["Price"] > 0)
].copy()

**Summary of sales_df**

In [10]:
print("Cleaned Transaction Rows:", len(clean_df))
print("Valid Sales Rows:", len(sales_df))
print("Removed Non-Sale Records:", len(clean_df) - len(sales_df))

Cleaned Transaction Rows: 1033031
Valid Sales Rows: 1007913
Removed Non-Sale Records: 25118


### **Create revenue**

In [11]:
sales_df["Revenue"] = (
    sales_df["Quantity"] *
    sales_df["Price"]
)

In [12]:
print("Negative revenue:", (sales_df["Revenue"] < 0).sum())
print("Zero revenue:", (sales_df["Revenue"] == 0).sum())

Negative revenue: 0
Zero revenue: 0


### **Extract date features**

In [13]:
sales_df["Year"] = sales_df["InvoiceDate"].dt.year
sales_df["Month"] = sales_df["InvoiceDate"].dt.month
sales_df["Day"] = sales_df["InvoiceDate"].dt.day
sales_df["DayOfWeek"] = sales_df["InvoiceDate"].dt.dayofweek
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour
sales_df["IsWeekend"] = (
    sales_df["DayOfWeek"] >= 5
)

### **Create daily sales dataset**

In [14]:
daily_sales = (
    sales_df
    .groupby(sales_df["InvoiceDate"].dt.date)
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique")
    )
    .reset_index()
)



**Rename**

In [15]:
daily_sales.rename(
    columns={"InvoiceDate": "Date"},
    inplace=True
)

**Convert**

In [16]:
daily_sales["Date"] = pd.to_datetime(
    daily_sales["Date"]
)

**Sort**

In [17]:
daily_sales = (
    daily_sales
    .sort_values("Date")
    .reset_index(drop=True)
)

**Check the result**

In [18]:
print(daily_sales.head())
print(daily_sales.tail())
print(daily_sales.shape)

        Date   Revenue  Quantity  Orders  Customers
0 2009-12-01  54351.23     26098     119         91
1 2009-12-02  63172.58     31804     115         94
2 2009-12-03  73972.45     49221     124        106
3 2009-12-04  40582.32     21210      89         76
4 2009-12-05   9803.05      5119      30         26
          Date    Revenue  Quantity  Orders  Customers
599 2011-12-05   88620.84     44497     127        105
600 2011-12-06   56558.83     30377     115        103
601 2011-12-07   75315.55     41789     106         94
602 2011-12-08   82371.55     35010     120        105
603 2011-12-09  200918.98     93950      44         35
(604, 5)


### **Check missing dates**

In [19]:
full_date_range = pd.date_range(
    start=daily_sales["Date"].min(),
    end=daily_sales["Date"].max(),
    freq="D"
)

missing_dates = full_date_range.difference(
    daily_sales["Date"]
)

print("Number of missing dates:", len(missing_dates))
print(missing_dates[:20])

Number of missing dates: 135
DatetimeIndex(['2009-12-12', '2009-12-19', '2009-12-24', '2009-12-25',
               '2009-12-26', '2009-12-27', '2009-12-28', '2009-12-29',
               '2009-12-30', '2009-12-31', '2010-01-01', '2010-01-02',
               '2010-01-03', '2010-01-09', '2010-01-16', '2010-01-23',
               '2010-01-30', '2010-02-06', '2010-02-13', '2010-02-20'],
              dtype='datetime64[ns]', freq=None)


In [20]:
missing_values = clean_df.isnull().sum()

print("Missing values:")
print(missing_values)

print("\n")

missing_summary = pd.DataFrame({
    "Missing Count": clean_df.isnull().sum(),
    "Missing Percentage": (
        clean_df.isnull().mean() * 100
    ).round(2)
})

print(missing_summary)

Missing values:
Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235146
Country             0
IsCancelled         0
dtype: int64


             Missing Count  Missing Percentage
Invoice                  0                0.00
StockCode                0                0.00
Description           4275                0.41
Quantity                 0                0.00
InvoiceDate              0                0.00
Price                    0                0.00
Customer ID         235146               22.76
Country                  0                0.00
IsCancelled              0                0.00


Missing product descriptions were retained because the corresponding transactions contained valid product codes and sales information.

### **Save processed data set**

In [21]:
clean_df.to_csv(
    "../data/processed/clean_transactions.csv",
    index=False
)

sales_df.to_csv(
    "../data/processed/valid_sales.csv",
    index=False
)

daily_sales.to_csv(
    "../data/processed/daily_sales.csv",
    index=False
)

In [22]:
clean_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False


In [23]:
sales_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,Revenue,Year,Month,Day,DayOfWeek,Hour,IsWeekend
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,2009,12,1,1,7,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,2009,12,1,1,7,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,2009,12,1,1,7,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,2009,12,1,1,7,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,2009,12,1,1,7,False


In [24]:
daily_sales.head()

,Date,Revenue,Quantity,Orders,Customers
0,2009-12-01,54351.23,26098,119,91
1,2009-12-02,63172.58,31804,115,94
2,2009-12-03,73972.45,49221,124,106
3,2009-12-04,40582.32,21210,89,76
4,2009-12-05,9803.05,5119,30,26
